In [167]:
import httpx

## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

In [48]:
from agents import Agent, WebSearchTool, trace, Runner, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
from IPython.display import display, Markdown
from messenger import send_email, push

In [49]:
load_dotenv(override=True)

True

In [122]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool,ModelSettings, OpenAIChatCompletionsModel, output_guardrail, GuardrailFunctionOutput
import os
from pydantic import BaseModel, Field

In [ ]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if  openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AQ
OpenRouter API Key exists and begins sk-or-
Groq API Key exists and begins gsk_


In [124]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

In [169]:
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

In [295]:
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.1-flash-lite", openai_client=gemini_client)
# kimi_model = OpenAIChatCompletionsModel(model="moonshotai/kimi-k2-instruct", openai_client=groq_client)
kimi_model = OpenAIChatCompletionsModel(model="moonshotai/kimi-k2.6", openai_client=openrouter_client, 
                                         )
oss_model = OpenAIChatCompletionsModel(model="openai/gpt-oss-120b", openai_client=groq_client)
llama_model = OpenAIChatCompletionsModel(model="llama-3.1-8b-instant", 
                                       openai_client=groq_client)
qwen_model = OpenAIChatCompletionsModel(model="qwen/qwen3.6-27b", openai_client=groq_client)

In [285]:
# use it with kimi as its a trillion params model
model_config = ModelSettings(
    extra_body={"max_tokens": 6000} 
)

In [296]:
# Constants 

MODEL_NAME = "gpt-5.4-mini"
MODEL_NAME_KIMI = kimi_model
MODEL_NAME_GEMINI = gemini_model
# MODEL_NAME_LLAMA = llama_model
MODEL_NAME_QWEN = qwen_model
MODEL_NAME_OSS = oss_model
USE_EMAIL = True
HOW_MANY_SEARCHES = 3

In [129]:
# sales_agent1 = Agent(name="Gemini Sales Agent", instructions=instructions, model=gemini_model)
# sales_agent2 = Agent(name="Kimi2 Sales Agent", instructions=instructions, model=kimi_model,
#                       model_settings=model_config )
# sales_agent3 = Agent(name="GPT-OSS Sales Agent",instructions=instructions, model=oss_model)

## Strategy for the Deep Research Agent

We are going to do it the bulletproof way.

We are going to orchestrate with code: separate calls to `Runner.run()` for each step in the process.

We will use Structured Outputs at each point.

## We will build 4 Agents:

1. The Search Agent: searches the web for information
2. The Planner Agent: given a question, comes up with a list of searches that should be made
3. The Writer Agent: writes a robust report
4. The Emailer Agent: crafts and sends an email

And then 4 python functions, 1 to call Runner.run() for each of the 4 agents.


## Agent 1: The Search Agent

### OpenAI Hosted Tools

https://openai.github.io/openai-agents-python/tools/#hosted-tools

A paid, quick approach to carrying out managed functionality on OpenAI's cloud.

Their docs surface these tools, but it's worth keeping in mind that they're costly and lock you in to the OpenAI ecosystem.

OpenAI offers the following hosted tools:

`WebSearchTool` lets an agent search the web.  
`FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
`CodeInterpreterTool` lets the LLM execute code in a sandboxed environment.  
`HostedMCPTool` exposes a remote MCP server's tools to the model.  
`ImageGenerationTool` generates images from a prompt.  
`ToolSearchTool` lets the model load deferred tools, namespaces, or hosted MCP servers on demand.  

### Important note - API charge of WebSearchTool

This currently costs 1 cent per call for OpenAI WebSearchTool. That can add up to about $1 for the next 2 labs. We'll use free and low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern. Also student Christian W. pointed out that OpenAI can sometimes charge for multiple searches for a single call, so it could sometimes cost more than 1 cent per call.

Costs are in the Tools section here: https://developers.openai.com/api/docs/pricing


In [130]:
from openai import OpenAI

In [131]:
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_api_key,
)

@function_tool
def openrouter_web_search(query: str) -> str:
    """Performs live web search via OpenRouter's server-side search tool."""
    response = openrouter_client.chat.completions.create(
        model="meta-llama/llama-3.3-70b-instruct",
        messages=[{"role": "user", "content": f"Search the web and summarize: {query}"}],
        tools=[{"type": "openrouter:web_search"}],
    )
    return response.choices[0].message.content

In [170]:
SEARXNG_BASE_URL = os.environ.get("SEARXNG_URL", "http://localhost:8080")

In [219]:
SEARXNG_BASE_URL = os.environ.get("SEARXNG_URL", "http://localhost:8080").rstrip("/")


@function_tool
async def search_searxng(query: str, max_results: int = 2) -> str:
    """Search SearXNG result snippets only.

    Do not open, fetch, follow, download, or read local or internet files, web pages,
    PDFs, documents, images, or result URLs. Use only this SearXNG HTTP endpoint.
    """
    params = {
        "q": query,
        "format": "json",
    }
    search_url = f"{SEARXNG_BASE_URL}/search"

    try:
        async with httpx.AsyncClient(timeout=10.0) as client:
            response = await client.get(search_url, params=params)
            response.raise_for_status()
            data = response.json()
    except httpx.ConnectError:
        return (
            f"SearXNG is unavailable at {search_url}. "
            "Start SearXNG or set SEARXNG_URL to a running instance."
        )
    except httpx.HTTPError as exc:
        return f"SearXNG request failed at {search_url}: {exc}"

    results = data.get("results", [])[:max_results]
    if not results:
        return "No results found."

    formatted = []
    for item in results:
        title = item.get("title", "No Title")
        url = item.get("url", "")
        content = item.get("content", "No content snippet")
        formatted.append(f"Title: {title}\nURL: {url}\nSnippet: {content}\n---")

    return "\n".join(formatted)

In [245]:
!uv pip install ddgs

Using Python 3.12.12 environment at: D:\ArtificialIntelligence\agents\.venv
Resolved 5 packages in 858ms
Prepared 1 package in 497ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 244ms
 + ddgs==9.16.0
d:\ArtificialIntelligence\agents\.venv\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedWriter name=3>
  res = process_handler(cmd, _system_body)
d:\ArtificialIntelligence\agents\.venv\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=4>
  res = process_handler(cmd, _system_body)
d:\ArtificialIntelligence\agents\.venv\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=5>
  res = process_handler(cmd, _system_body)


In [261]:
from ddgs import DDGS
from agents import function_tool

@function_tool
def search_web_duckduckgo(query: str, max_results: int = 3) -> str:
    """Searches the web using DuckDuckGo.

    Args:
        query: Search keywords.
        max_results: Max results to return.
    """
    try:
        results = DDGS().text(query, max_results=max_results)
        if not results:
            return "No results found."
        
        formatted = []
        for item in results:
            title = item.get("title", "No Title")
            url = item.get("href", "")
            snippet = item.get("body", "")
            formatted.append(f"Title: {title}\nURL: {url}\nSnippet: {snippet}\n---")
        return "\n".join(formatted)
    except Exception as exc:
        return f"Error connecting to DuckDuckGo: {exc}"

In [262]:
INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and 
produce a concise summary of the results. The summary must be 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""
task = "UP 2027 elections analysis and predictions"

settings = ModelSettings(
    tool_choice="required",
    max_tokens=1000,
)
# tools = [openrouter_web_search]
# tools=[{"type": "openrouter:web_search"}]
tools = [WebSearchTool()]

In [263]:
search_agent = Agent(
    name="Search Agent with duckduckgo",
    instructions=INSTRUCTIONS,
    model=MODEL_NAME_GEMINI,
    tools=[search_web_duckduckgo],
)

In [179]:
# search_agent = Agent(name="Search Agent", instructions=INSTRUCTIONS, tools=tools, model=MODEL_NAME, model_settings=settings)

In [264]:
from IPython.display import Markdown, display

In [265]:
result = await Runner.run(search_agent, task)
display(Markdown(result.final_output))

The upcoming 2027 Uttar Pradesh Legislative Assembly election is shaping up to be a critical litmus test for the ruling Bharatiya Janata Party (BJP) and Chief Minister Yogi Adityanath. After a decade of governance, the BJP faces the complex challenge of managing anti-incumbency sentiments while striving to maintain its stronghold in the country’s most populous state. The political atmosphere has been intensified by the recent performance of the opposition, particularly the bolstered Samajwadi Party-Congress alliance, which has gained momentum following recent electoral gains and by-poll outcomes.

Analysts view the 2027 contest as a pivotal moment that could significantly reshape Indian national politics. Key factors influencing the landscape include the effectiveness of the BJP’s organizational machinery, the consolidation of opposition vote banks, and evolving caste-based socio-political dynamics. As the major political players begin recalibrating their strategies, the election is expected to be a high-stakes battle centered on governance performance, welfare delivery, and the competing narratives of national development versus localized social justice.

### As always, take a look at the trace

https://platform.openai.com/traces

## Agent 2: The Planner Agent

### We will now use Structured Outputs, and include a description of the fields

In [252]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

In [ ]:
WebSearchPlan.model_json_schema()

{'$defs': {'WebSearchItem': {'properties': {'reason': {'description': 'Your reasoning for why this search is important to the query.',
     'title': 'Reason',
     'type': 'string'},
    'query': {'description': 'The search term to use for the web search.',
     'title': 'Query',
     'type': 'string'}},
   'required': ['reason', 'query'],
   'title': 'WebSearchItem',
   'type': 'object'}},
 'properties': {'searches': {'description': 'A list of web searches to perform to best answer the query.',
   'items': {'$ref': '#/$defs/WebSearchItem'},
   'title': 'Searches',
   'type': 'array'}},
 'required': ['searches'],
 'title': 'WebSearchPlan',
 'type': 'object'}

In [297]:
# See note above about cost of WebSearchTool

INSTRUCTIONS = f"""
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for.
"""

planner_agent = Agent(name="Planner Agent", instructions=INSTRUCTIONS, model=MODEL_NAME_QWEN, output_type=WebSearchPlan)

In [298]:
result = await Runner.run(planner_agent, task)
searches_markdown = "\n".join(
    f"- **{item.query}**: {item.reason}"
    for item in result.final_output.searches
)
display(Markdown(searches_markdown))

- **Uttar Pradesh 2027 assembly election analysis predictions**: Directly targets the user's query to find comprehensive articles, expert opinions, and data-driven forecasts about the upcoming UP elections.
- **UP 2027 elections seat projection swing analysis BJP SP BSP**: Focuses on quantitative predictions, seat-wise forecasts, and party-wise trends involving key players, which are crucial for election predictions.
- **factors influencing Uttar Pradesh 2027 elections economy development caste equation**: Captures the underlying socio-economic and political dynamics that analysts and experts cite when making predictions, ensuring a deeper understanding of the analysis component.

## Agent 3: The Writer Agent

In [255]:
INSTRUCTIONS = """
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed. Aim
for 5-10 pages of content, at least 1000 words.
Return only the Markdown report, without JSON or code fences.
"""


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(name="Writer Agent", instructions=INSTRUCTIONS, model=MODEL_NAME_OSS)

## Agent 4: The email agent

In [ ]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects.
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")
    return "Email sent successfully"

In [ ]:
send_email_tool.params_json_schema

{'properties': {'subject': {'description': 'The subject of the email',
   'title': 'Subject',
   'type': 'string'},
  'text_body': {'description': 'The body of the email as plain text',
   'title': 'Text Body',
   'type': 'string'},
  'html_body': {'description': 'The HTML body of the email',
   'title': 'Html Body',
   'type': 'string'}},
 'required': ['subject', 'text_body', 'html_body'],
 'title': 'send_email_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [274]:
INSTRUCTIONS = """
You are provided with a detailed report. Use your tool to send an email, converting the report into
a clean, well presented HTML email with an appropriate subject line.
Make it look neat and tidy with tables and bullet points. Use HTML formatting for the email body.
"""

email_agent = Agent(name="Email Agent", instructions=INSTRUCTIONS, tools=[send_email_tool], model=MODEL_NAME_GEMINI)

## Now to Orchestrate by Code

The next 2 functions will plan and execute the search, using the Agents, with calls to `Runner.run()`

In [271]:
async def run_searches(query: str):
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    searches = result.final_output.searches
    print(f"Will perform {len(searches)} searches")
    tasks = [search(item) for item in searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results


async def search(item: WebSearchItem):
    input_message = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input_message)
    return result.final_output

The next 2 functions write a report and email it

In [272]:
from html import escape


async def write_report(query: str, search_results: list[str]):
    print("Thinking about report...")
    input_message = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input_message)
    print("Finished writing report")
    return ReportData(
        short_summary=result.final_output[:500],
        markdown_report=result.final_output,
        follow_up_questions=[],
    )


async def send_report_email(report: ReportData):
    print("Writing email...")
    subject = "AI Agent Frameworks Research Report"
    text_body = report.markdown_report
    html_body = f"<html><body><pre>{escape(report.markdown_report)}</pre></body></html>"
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")
    print("Email sent")
    return "Email sent successfully"

### Showtime!

In [299]:
query ="A detailed report on the outlook for the 2027 Uttar Pradesh elections, including political parties, candidates, and key issues."

with trace("Research trace with OpenAI and DuckDuckGo"):
    print("Starting research...")
    search_results = await run_searches(query)
    report = await write_report(query, search_results)
    await send_report_email(report)  
    print("Hooray!")

Starting research...
Planning searches...
Will perform 3 searches
Finished searching
Thinking about report...
Finished writing report
Writing email...
Email sent
Hooray!


### As always, take a look at the trace

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Congratulations on your progress, and a request</h2>
            <span style="color:#00cc00;">You've reached an important moment with the course; you've created a valuable Agent using one of the latest Agent frameworks. You've upskilled, and unlocked new commercial possibilities. Take a moment to celebrate your success!<br/><br/>Something I should ask you -- my editor would smack me if I didn't mention this. If you're able to rate the course on Udemy, I'd be seriously grateful: it's the most important way that Udemy decides whether to show the course to others and it makes a massive difference.<br/><br/>And another reminder to <a href="https://www.linkedin.com/in/eddonner/">connect with me on LinkedIn</a> if you wish! If you wanted to post about your progress on the course, please tag me and I'll weigh in to increase your exposure.
            </span>
        </td>
    </tr>